### GNSS-InSAR Calibration Workflow

This notebook examplify how to use the Venti package to calibrate OPERA DISP-S1 displacement 
products using GNSS reference data from the University of Nevada, Reno (UNR) grid timeseries.

Workflow steps:
1. Download GNSS stations within the InSAR scene bounds
2. Compute a GNSS LOS velocity or displacement field
3. Correct unwrapping errors relative to the GNSS reference
4. Estimate and remove a long-wavelength calibration surface via windowed plane fitting

Two GNSS `grid_type` modes are supported:
- **`constant`** — fits a single GNSS velocity field and scales it per epoch
- **`variable`** — computes epoch-specific GNSS displacements

### Imports

In [ ]:
import logging
from pathlib import Path

import numpy as np
import rasterio as rio
from tqdm import tqdm

from venti.gnss.reference import GNSSReference
from venti.io.raster import get_bounds, read_geotiff, read_netcdf, write_geotiff
from venti.spatial.processor import SpatialProcessor
from venti.unwrap import correct_region_offset
from venti.workflow.utils import (
    compute_average_temporal_coherence,
    downsample_array,
    match_correction_to_displacement,
    parse_window_size_meters,
    upsample_array,
    get_file_dates,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)-20s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [ ]:
%load_ext autoreload
%autoreload 2

### Input configuration

Set the paths and parameters for the dataset. All paths should point to existing files or directories.

In [ ]:
# Input paths
input_dir = Path("/u/aurora-r0/cabrera/opera_vlm/data/36540/disp") # Path("/path/to/disp_nc_files")   # directory of OPERA *.nc files
los_file  = Path("/u/aurora-r0/cabrera/opera_vlm/data/36540/static_layer") # Path("/path/to/los_enu.tif")      # 3-band GeoTIFF: east, north, up
mask_file = Path("/u/aurora-r0/cabrera/opera_vlm/data/36540/aligned_water_mask.tif") # Path("/path/to/water_mask.tif")   # binary mask (1 = valid land)
output_dir = Path("/u/aurora-r0/cabrera/opera_vlm/data/36540/venti_test") # Path("/path/to/output")          # results directory
tropo_dir  = None                             # or Path("/path/to/tropo_corrections")

## Processing selections. Use one of the three options below:

# Option A: single file
input_files = Path("/path/to/single_file.nc")

# Option B: explicit list of files
# input_files = [
#     Path("/path/to/file_a.nc"),
#     Path("/path/to/file_b.nc"),
# ]

# Option C: full directory (None => discover all *.nc from input_dir)
# input_files = None

# Limit to the first N files when testing (None => process all)
max_files: int | None = None

# Calibration parameters
grid_type         = "constant"   # "constant" (velocity) or "variable" (per-epoch)
reference_frame   = "IGS20"      # or "IGS14"
downsample_factor = 1            # set > 1 to speed up the plane fit
window_meters     = 30_000       # plane-fitting window size in metres
posting_meters    = 30           # pixel spacing in metres (OPERA default)
start_year        = 2014.0       # earliest year used for velocity estimation

output_dir.mkdir(parents=True, exist_ok=True)
gnss_dir = output_dir / "GNSS"
gnss_dir.mkdir(exist_ok=True)

### Go over the available displacement files and match tropospheric corrections (if available)

In [ ]:
# Resolve displacement file list from whichever input option was set above.
if input_files is None:
    disp_files = sorted(input_dir.glob("*.nc"))
elif isinstance(input_files, (str, Path)):
    disp_files = [Path(input_files)]
else:
    disp_files = sorted(Path(f) for f in input_files)

if not disp_files:
    raise FileNotFoundError(f"No NetCDF files found — check input_files / input_dir")

if max_files is not None:
    disp_files = disp_files[:max_files]

print(f"Processing {len(disp_files)} displacement file(s)")
for f in disp_files:
    print(f"  {f.name}")

if tropo_dir is not None:
    tropo_files = sorted(tropo_dir.glob("tropo_corr*.tif"))
    matched_files = match_correction_to_displacement(tropo_files, disp_files)
    print(f"\nMatched {len(matched_files)} tropospheric correction files")
else:
    matched_files = match_correction_to_displacement(None, disp_files)

for tropo, disp in matched_files[:3]:
    print(f"  tropo={tropo}  disp={disp.name}")

### Average temporal coherence and reference point

The reference point is the pixel held at zero displacement throughout the stack.
It is chosen as the most coherent pixel inside the valid mask.

In [ ]:
coherence_path = compute_average_temporal_coherence(disp_files, output_dir)
print(f"Average coherence map: {coherence_path}")

In [ ]:
from opera_utils.disp import rebase_reference

reference_point = rebase_reference.find_reference_point(coherence_path)
refy, refx = reference_point
print(f"Reference point: row={refy}, col={refx}")

### Load mask and read scene bounds

In [ ]:
mask_data, mask_geo = read_geotiff(mask_file)
mask = mask_data.astype(bool)

# UTM EPSG extracted from the mask CRS
from pyproj import CRS
utm_epsg = CRS.from_user_input(mask_geo["crs"]).to_epsg()
print(f"Mask shape: {mask.shape}  |  UTM EPSG: {utm_epsg}")

# Scene bounds in UTM (S, N, W, E)
snwe = get_bounds(disp_files[0], as_latlon=False)
S, N, W, E = snwe
print(f"Bounds (UTM): S={S:.0f}  N={N:.0f}  W={W:.0f}  E={E:.0f}")

### Download GNSS data

`GNSSReference` handles the full download pipeline:
- fetches the UNR grid lookup table
- filters stations within the scene bounds
- downloads individual station timeseries files

In [ ]:
gnss = GNSSReference(
    bounds=snwe,
    output_dir=gnss_dir,
    reference_frame=reference_frame,
    utm_epsg=utm_epsg,
)
n_stations = gnss.download_stations()
print(f"Downloaded {n_stations} GNSS station files")

### Load LOS unit vectors

The LOS file is a 3-band GeoTIFF with the east, north, and up unit-vector components at each pixel.

In [ ]:
with rio.open(los_file) as src:
    de = src.read(1).astype(np.float32)   # east
    dn = src.read(2).astype(np.float32)   # north
    dv = src.read(3).astype(np.float32)   # vertical (up)

print(f"LOS arrays: shape={de.shape}  de range=[{de.min():.3f}, {de.max():.3f}]")

### Compute GNSS LOS reference field

Two modes are available, controlled by `grid_type`:

##### Constant mode (`grid_type = "constant"`)
A single LOS **velocity field** (m/yr) is estimated once from the full station timeseries
and interpolated onto the raster grid. Each epoch's GNSS reference is obtained by scaling
this field by the epoch time span.

##### Variable mode (`grid_type = "variable"`)
An epoch-specific LOS **displacement field** is computed for every epoch by reading the
station timeseries directly around `(ref_date, sec_date)`.

In [ ]:
if grid_type == "constant":
    print("Computing GNSS LOS velocity field (RBF interpolation)...")
    gnss_los_velocity = gnss.compute_velocity_los(
        los_east=de,
        los_north=dn,
        los_up=dv,
        netcdf_file=disp_files[0],
        start_year=start_year,
        method="rbf",
    )
    # Reference to zero at the reference point
    gnss_los_velocity -= gnss_los_velocity[refy, refx]

    # Optionally save for inspection
    np.save(output_dir / "gnss_los_velocity.npy", gnss_los_velocity)
    print(f"Velocity range: [{gnss_los_velocity.min():.4f}, {gnss_los_velocity.max():.4f}] m/yr")

In [ ]:
if grid_type == "variable":
    # Compute epoch-0 displacement as a preview — the full per-epoch computation
    # happens inside the calibration loop.
    print("Computing GNSS LOS displacement for epoch 0 (variable mode preview)...")
    _ref_0, _sec_0 = get_file_dates(disp_files[0])
    gnss_los_displacement_0 = gnss.compute_displacement_los(
        ref_date=_ref_0,
        sec_date=_sec_0,
        los_east=de,
        los_north=dn,
        los_up=dv,
        netcdf_file=disp_files[0],
        method="rbf",
    )
    gnss_los_displacement_0 -= gnss_los_displacement_0[refy, refx]
    gnss_los_displacement_0 *= 1000.0  # m -> mm
    print(
        f"Epoch 0 ({_ref_0:.3f} -> {_sec_0:.3f}): "
        f"LOS range [{gnss_los_displacement_0.min():.2f}, {gnss_los_displacement_0.max():.2f}] mm"
    )

#### Quick look at the GNSS LOS field (epoch 0 preview)

In [ ]:
import matplotlib.pyplot as plt

ny, nx = de.shape
S, N, W, E = snwe

# Map station UTM coordinates to pixel row/col indices
sta_row = (N - gnss.station_gdf.geometry.y.values) / (N - S) * ny
sta_col = (gnss.station_gdf.geometry.x.values - W) / (E - W) * nx

if grid_type == "constant":
    data_mm  = gnss_los_velocity * 1000.0
    cb_label = "LOS velocity (mm/yr)"
    title    = "GNSS LOS velocity field"
elif grid_type == "variable":
    data_mm  = gnss_los_displacement_0
    cb_label = "LOS displacement (mm)"
    _ref_0, _sec_0 = get_file_dates(disp_files[0])
    title    = f"GNSS LOS displacement — epoch 0 ({_ref_0:.3f} to {_sec_0:.3f})"

_vmax = np.nanpercentile(np.abs(data_mm), 98)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(data_mm, cmap="RdBu_r", vmin=-_vmax, vmax=_vmax)
ax.scatter(
    sta_col, sta_row,
    c="k", s=20, marker="^", edgecolors="white", linewidths=0.5,
    label=f"GNSS stations (n={len(gnss.station_gdf)})",
    zorder=5,
)
plt.colorbar(im, ax=ax, label=cb_label)
ax.set_title(title)
ax.set_xlabel("Column")
ax.set_ylabel("Row")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Visual Inspection before running the full calibration loop of the first 
# InSAR displacement in the stack, GNSS LOS field, and residual. 

if matched_files:
    _tropo_0, _disp_file_0 = matched_files[0]
    _ref_date_0, _sec_date_0 = get_file_dates(_disp_file_0)

    _disp_0, _, _ = read_netcdf(_disp_file_0, variable="displacement")
    _disp_0 = _disp_0 * 1000.0  # m -> mm
    _disp_0 -= _disp_0[refy, refx]

    if grid_type == "constant":
        _gnss_0 = gnss_los_velocity * (_ref_date_0 - _sec_date_0) * 1000.0
    else:
        _gnss_0 = gnss.compute_displacement_los(
            ref_date=_ref_date_0,
            sec_date=_sec_date_0,
            los_east=de,
            los_north=dn,
            los_up=dv,
            netcdf_file=_disp_file_0,
            method="rbf",
        ) * 1000.0
        _gnss_0 -= _gnss_0[refy, refx]

    _residual_0  = np.where(mask, _disp_0 - _gnss_0, np.nan)
    _vmax_0      = np.nanpercentile(np.abs(np.where(mask, _disp_0, np.nan)), 98)
    _rvmax_0     = np.nanpercentile(np.abs(_residual_0), 98)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(np.where(mask, _disp_0, np.nan), cmap="RdBu_r", vmin=-_vmax_0, vmax=_vmax_0)
    axes[0].set_title("InSAR displacement (mm)")

    axes[1].imshow(np.where(mask, _gnss_0, np.nan), cmap="RdBu_r", vmin=-_vmax_0, vmax=_vmax_0)
    axes[1].set_title("GNSS LOS (mm)")

    axes[2].imshow(_residual_0, cmap="RdBu_r", vmin=-_rvmax_0, vmax=_rvmax_0)
    axes[2].set_title("InSAR - GNSS residual (mm)")

    for ax in axes:
        ax.set_xlabel("Column")
        ax.set_ylabel("Row")

    plt.suptitle(f"Pre-calibration epoch 0: {_disp_file_0.name}", fontsize=9)
    plt.tight_layout()
    plt.show()

### Per-epoch calibration loop

For each displacement epoch:
1. Read the displacement and reference it to zero at `reference_point`
2. Subtract tropospheric correction if available
3. Compute the GNSS LOS reference for this epoch
4. Correct unwrapping errors with `correct_region_offset`
5. Fit and subtract a windowed polynomial calibration surface
6. Write the corrected displacement to GeoTIFF

In [ ]:
win_size = parse_window_size_meters(window_meters, posting_meters=posting_meters)
print(f"Window: {window_meters/1000:.0f} km = {win_size} pixels at {posting_meters} m/px")

processor = SpatialProcessor()
cal_surface_debug = None  # populated from the first calibrated epoch for post-loop inspection

In [ ]:
suffix = f"_corrected_{grid_type}_{reference_frame.lower()}"
if downsample_factor > 1:
    suffix += f"_downsample{downsample_factor}"
if tropo_dir is not None:
    suffix += "_tropo"

output_files = []

for tropo_file, disp_file in tqdm(matched_files, desc="Calibrating"):
    _is_first_epoch = len(output_files) == 0

    # Dates
    ref_date, sec_date = get_file_dates(disp_file)

    # Read displacement (convert m -> mm)
    disp, _, geo_info = read_netcdf(disp_file, variable="displacement")
    disp = disp * 1000.0
    disp -= disp[refy, refx]

    # Tropospheric correction (optional)
    if tropo_file is not None:
        tropo_data, _ = read_geotiff(tropo_file)
        tropo_mm = tropo_data * 1000.0
        tropo_mm -= tropo_mm[refy, refx]
        disp -= tropo_mm

    # GNSS LOS for this epoch
    if grid_type == "constant":
        # Scale velocity (m/yr -> mm) by the epoch time span
        gnss_los = gnss_los_velocity * (ref_date - sec_date) * 1000.0
    else:
        # Epoch-specific displacement directly from station files
        gnss_los = gnss.compute_displacement_los(
            ref_date=ref_date,
            sec_date=sec_date,
            los_east=de,
            los_north=dn,
            los_up=dv,
            netcdf_file=disp_file,
            method="rbf",
        )
        gnss_los *= 1000.0   # m -> mm
        gnss_los -= gnss_los[refy, refx]

    # Apply valid-pixel mask
    disp_mask  = np.ma.masked_invalid(disp).mask
    valid_mask = (~disp_mask) & mask
    disp = np.where(valid_mask, disp, np.nan)

    # Unwrap error correction
    disp = correct_region_offset(input_disp=disp, gnss_los=gnss_los, mask=mask)

    if isinstance(disp, np.ma.MaskedArray):
        disp = disp.filled(np.nan)
    if isinstance(gnss_los, np.ma.MaskedArray):
        gnss_los = gnss_los.filled(np.nan)

    # Optional downsampling before plane fit
    original_shape = disp.shape
    if downsample_factor > 1:
        disp_ds     = downsample_array(disp,     downsample_factor)
        gnss_los_ds = downsample_array(gnss_los, downsample_factor)
        win_size_ds = max(1, win_size // downsample_factor)
    else:
        disp_ds, gnss_los_ds, win_size_ds = disp, gnss_los, win_size

    # Windowed polynomial calibration surface
    cal_surface = processor.fit_windowed_surface(
        insar_data=disp_ds,
        gnss_los=gnss_los_ds,
        bounds=snwe,
        window_size_x=win_size_ds,
        window_size_y=win_size_ds,
        window_overlap_x=10,
        window_overlap_y=10,
        poly_order=1.5,
        n_jobs=-1,
    )

    # Save epoch-0 data for the post-loop inspection plots
    if _is_first_epoch:
        cal_surface_debug = (
            upsample_array(cal_surface, original_shape)
            if downsample_factor > 1
            else cal_surface.copy()
        )
        disp_debug     = disp.copy()
        gnss_los_debug = gnss_los.copy()

    corrected_ds = disp_ds - cal_surface

    # Upsample back to original resolution and convert to metres
    if downsample_factor > 1:
        corrected = upsample_array(corrected_ds, original_shape)
    else:
        corrected = corrected_ds

    corrected_m = corrected / 1000.0

    # Write output
    out_file = output_dir / f"{disp_file.stem}{suffix}.tif"
    write_geotiff(
        corrected_m,
        out_file,
        reference_file=disp_file,
        nodata=np.nan,
        descriptions=[f"Calibrated displacement ({grid_type} GNSS, {reference_frame})"],
    )
    output_files.append(out_file)

print(f"\nWrote {len(output_files)} calibrated files to {output_dir}")

### Inspect calibration 

In [ ]:
if not output_files:
    raise RuntimeError("No output files were produced — check the loop above.")

corrected_m, _ = read_geotiff(output_files[0])
corrected_mm = corrected_m * 1000.0

# Use the epoch-0 debug arrays saved during the loop
insar_gnss_residual = np.where(mask, disp_debug - gnss_los_debug, np.nan)

_vmax  = np.nanpercentile(np.abs(np.where(mask, disp_debug, np.nan)), 98)
_rvmax = np.nanpercentile(np.abs(insar_gnss_residual), 98)

fig, axes = plt.subplots(1, 5, figsize=(28, 5))

axes[0].imshow(np.where(mask, disp_debug, np.nan), cmap="RdBu_r", vmin=-_vmax, vmax=_vmax)
axes[0].set_title("InSAR displacement (mm)")

axes[1].imshow(np.where(mask, gnss_los_debug, np.nan), cmap="RdBu_r", vmin=-_vmax, vmax=_vmax)
axes[1].set_title("GNSS LOS (mm)")

axes[2].imshow(insar_gnss_residual, cmap="RdBu_r", vmin=-_rvmax, vmax=_rvmax)
axes[2].set_title("InSAR - GNSS residual (mm)")

if cal_surface_debug is not None:
    _csmax = np.nanpercentile(np.abs(cal_surface_debug), 98)
    axes[3].imshow(cal_surface_debug, cmap="RdBu_r", vmin=-_csmax, vmax=_csmax)
    axes[3].set_title("Calibration surface (mm)")
else:
    axes[3].set_visible(False)

axes[4].imshow(np.where(mask, corrected_mm, np.nan), cmap="RdBu_r", vmin=-_vmax, vmax=_vmax)
axes[4].set_title("Calibrated InSAR (mm)")

for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")

plt.suptitle(f"Epoch 0: {output_files[0].name}", fontsize=9)
plt.tight_layout()
plt.show()

### Process Summary

In [ ]:
print("Calibration complete")
print(f"  grid_type        : {grid_type}")
print(f"  reference_frame  : {reference_frame}")
print(f"  GNSS stations    : {n_stations}")
print(f"  reference_point  : row={refy}, col={refx}")
print(f"  window           : {window_meters/1000:.0f} km ({win_size} px)")
print(f"  downsample       : {downsample_factor}x")
print(f"  files processed  : {len(output_files)} / {len(disp_files)}")
print(f"  output_dir       : {output_dir}")